In [2]:
from __future__ import annotations

import os
import re
import unicodedata
import difflib
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Tuple, List
import sys 


import numpy as np
import pandas as pd
from os.path import join
# --- Logging
parent_dir = Path(os.getcwd()).parents[0]          # project/
LOG_FILE = join(parent_dir, "logs.log")
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s: %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.FileHandler(LOG_FILE, encoding="utf-8"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

# --- Project paths helper
sys.path.append(str(parent_dir))
from src.paths import all_dirs  # must exist in your repo
dirs = all_dirs()

# =========================
# Inputs (adjust paths)
# =========================
POWERPLANTS_XLSX = join(dirs["data/processed/generation"], "cleaned_generation_2022.xlsx")
CAUDALS_CSV = join(dirs["data/raw/generation"], "caudals.csv")

YEAR = 2022
OUTPUT_DIR = os.path.join(dirs["data/processed/generation"], "hidro_max_profiles")
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: manual alias file (if it exists, it will be used)
# Expected columns: central_id, caudal_id
ALIASES_CSV = OUTPUT_DIR / "aliases_central_to_caudal.csv"


# =========================
# Logging
# =========================
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("pmax_builder")


# =========================
# Model configuration
# =========================
@dataclass(frozen=True)
class ModelConfig:
    q_rated_quantile_pasada: float = 0.95
    q_rated_quantile_embalse: float = 0.95
    storage_months_embalse: int = 3
    monthly_anchor: str = "mid"          # "mid" or "start"
    interpolation_method: str = "time"   # pandas interpolation method

    fuzzy_cutoff: float = 0.72

    # Nearest-anchor confidence thresholds
    nn_high_km: float = 40.0
    nn_med_km: float = 120.0
    nn_gap_high_km: float = 20.0
    nn_gap_med_km: float = 10.0


CFG = ModelConfig()


# =========================
# Rule-based matching configuration
# =========================
# Ordered list: first match wins (higher priority first).
# Each rule checks tokens in central_id_norm and central_norm.
RULES_TOKEN_MATCH: List[Tuple[List[str], str]] = [
    # Paute / Amaluza system
    (["lateral", "amaluza"], "lateral_amaluza"),
    (["ingreso", "amaluza"], "ingreso_amaluza"),
    (["amaluza"], "ingreso_amaluza"),
    (["molino"], "ingreso_amaluza"),
    (["sopladora"], "ingreso_amaluza"),
    (["paute"], "ingreso_amaluza"),
    # Mazar (upstream storage)
    (["mazar"], "mazar"),
    # Daule/Peripa complex
    (["daule", "peripa"], "daule_peripa"),
    (["marcel", "laniado"], "daule_peripa"),
    (["marcel_laniado"], "daule_peripa"),
    (["baba"], "daule_peripa"),
    # Coca Codo / Quijos / Napo
    (["coca", "sinclair"], "coca_codo_sinclair"),
    (["coca"], "coca_codo_sinclair"),
    (["sinclair"], "coca_codo_sinclair"),
    (["quijos"], "coca_codo_sinclair"),
    # Pastaza / Agoyan / San Francisco (best-effort proxy)
    (["agoyan"], "agoyan"),
    (["pastaza"], "agoyan"),
    (["san", "francisco"], "agoyan"),
    (["san_francisco"], "agoyan"),
    # Others that exist as series
    (["delsitanisagua"], "delsitanisagua"),
    (["pisayambo"], "pisayambo"),
]


# =========================
# Utilities
# =========================
def normalize_id(x: object) -> str:
    """
    Normalize strings to ASCII lowercase with underscores.
    Example: Coca Codo Sinclair -> coca_codo_sinclair
    """
    s = "" if x is None else str(x)
    s = s.strip()
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


def token_set(norm_text: str) -> set[str]:
    """
    Split a normalized id into tokens.
    """
    if not norm_text:
        return set()
    return {t for t in norm_text.split("_") if t}


def find_col(df: pd.DataFrame, target: str) -> str:
    """
    Find a column name matching 'target' (case-insensitive, stripped).
    """
    tgt = target.lower()
    for c in df.columns:
        if str(c).strip().lower() == tgt:
            return c
    raise KeyError(f"Missing required column: {target}")


def find_first_col(df: pd.DataFrame, targets: List[str]) -> Optional[str]:
    """
    Find the first matching column among a list of targets (case-insensitive).
    Returns None if no match.
    """
    cols = {str(c).strip().lower(): c for c in df.columns}
    for t in targets:
        if t.lower() in cols:
            return cols[t.lower()]
    return None


def is_hydro_row(row: pd.Series, col_tipo: str, col_tech: str) -> bool:
    """
    Decide whether a row represents a hydro plant.
    """
    tipo = str(row.get(col_tipo, "")).strip().lower()
    tech = str(row.get(col_tech, "")).strip().lower()
    return (tipo == "hidraulica") or ("hidraulica" in tech)


def monthly_index(year: int, anchor: str = "mid") -> pd.DatetimeIndex:
    """
    Build a monthly timestamp index.
    - anchor="start": month starts (MS)
    - anchor="mid": approximate mid-month timestamps (better representation for monthly means)
    """
    if anchor == "start":
        return pd.date_range(f"{year}-01-01", f"{year}-12-01", freq="MS")

    idx = []
    for m in range(1, 13):
        start = pd.Timestamp(year, m, 1)
        days = start.days_in_month
        idx.append(start + pd.Timedelta(hours=12) + pd.Timedelta(days=days / 2))
    return pd.DatetimeIndex(idx)


def monthly_to_hourly(monthly_vals: pd.Series, year: int, cfg: ModelConfig) -> pd.Series:
    """
    Interpolate monthly values to hourly resolution for a given year.
    """
    idx_m = monthly_index(year, anchor=cfg.monthly_anchor)
    s = pd.Series(monthly_vals.values, index=idx_m).sort_index()

    hourly_idx = pd.date_range(
        f"{year}-01-01 00:00:00",
        f"{year}-12-31 23:00:00",
        freq="h",
    )

    s2 = s.reindex(s.index.union(hourly_idx)).sort_index()
    s2 = s2.interpolate(method=cfg.interpolation_method)
    s2 = s2.reindex(hourly_idx).bfill().ffill()
    return s2


def parse_caudals(path: Path) -> pd.DataFrame:
    """
    Load monthly inflows from CSV (robust to "no header" files).

    Supported formats:
    - With header:
        name,1,2,...,12
      or an explicit name column ("central"/"rio"/"river") + month columns
    - Without header:
        daule_peripa,324,654,...,29

    Returns:
        DataFrame indexed by normalized series id, columns "1".."12" (strings).
    """
    df = pd.read_csv(path)

    def _has_month_columns(cols: List[str]) -> bool:
        s = {str(c).strip().lower() for c in cols}
        return all(str(i) in s for i in range(1, 13))

    # If the file has no header, pandas will treat the first row as header.
    # Detect this by checking whether columns include 1..12 (or month names).
    if not _has_month_columns(list(df.columns)):
        # Try re-reading with header=None and enforce expected structure
        df = pd.read_csv(path, header=None)
        if df.shape[1] != 13:
            raise ValueError(
                f"Unexpected CAUDALS.csv shape after header=None: {df.shape}. "
                f"Expected 13 columns (name + 12 months)."
            )
        df.columns = ["name"] + [str(i) for i in range(1, 13)]
        df = df.set_index("name")
    else:
        # Decide index (header mode)
        first = str(df.columns[0])
        cols_lower = [str(c).lower() for c in df.columns]
        if first.lower().startswith("unnamed") or ("central" in cols_lower) or ("rio" in cols_lower) or ("river" in cols_lower):
            if "central" in cols_lower:
                name_col = df.columns[cols_lower.index("central")]
                df = df.set_index(name_col)
            elif "rio" in cols_lower:
                name_col = df.columns[cols_lower.index("rio")]
                df = df.set_index(name_col)
            elif "river" in cols_lower:
                name_col = df.columns[cols_lower.index("river")]
                df = df.set_index(name_col)
            else:
                df = df.set_index(df.columns[0])
        else:
            # If the first column is a string id column but not named, set it as index
            # Example: columns are ["name","1",...,"12"] already; do nothing if "1".."12" exist
            pass

        # Keep only month columns 1..12
        df = df[[str(i) for i in range(1, 13)]].copy()

    df = df.apply(pd.to_numeric, errors="coerce")
    df.index = df.index.map(normalize_id)
    return df


def best_fuzzy_match(query: str, candidates: List[str], cutoff: float = 0.72) -> Tuple[Optional[str], float]:
    """
    Return best fuzzy match from candidates using difflib.
    """
    if not candidates:
        return None, 0.0
    m = difflib.get_close_matches(query, candidates, n=1, cutoff=cutoff)
    if not m:
        return None, 0.0
    score = difflib.SequenceMatcher(None, query, m[0]).ratio()
    return m[0], float(score)


def load_aliases(path: Path) -> Dict[str, str]:
    """
    Load manual aliases mapping plant central_id -> caudal series id.
    """
    if not path.exists():
        return {}
    df = pd.read_csv(path)
    c1 = find_col(df, "central_id")
    c2 = find_col(df, "caudal_id")

    out: Dict[str, str] = {}
    for _, r in df.iterrows():
        out[normalize_id(r[c1])] = normalize_id(r[c2])
    return out


def pmax_pu_pasada(Qm: pd.Series, cfg: ModelConfig) -> pd.Series:
    """
    Run-of-river model:
    p_max_pu = clip(Q / Q_rated, 0, 1),
    with Q_rated estimated as a quantile of monthly inflow.
    """
    qr = float(Qm.quantile(cfg.q_rated_quantile_pasada))
    if not np.isfinite(qr) or qr <= 0:
        return pd.Series(np.nan, index=Qm.index)
    return (Qm / qr).clip(0.0, 1.0)


def pmax_pu_embalse(Qm: pd.Series, cfg: ModelConfig) -> pd.Series:
    """
    Reservoir model (simple smoothing approach):
    - Compute an effective inflow as rolling mean over a storage window (months)
    - p_max_pu = clip(Q_eff / Q_rated, 0, 1),
      with Q_rated estimated as a quantile of Q_eff
    """
    Qeff = Qm.rolling(cfg.storage_months_embalse, min_periods=1).mean()
    qr = float(Qeff.quantile(cfg.q_rated_quantile_embalse))
    if not np.isfinite(qr) or qr <= 0:
        return pd.Series(np.nan, index=Qm.index)
    return (Qeff / qr).clip(0.0, 1.0)


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """
    Great-circle distance between two points (degrees) in kilometers.
    """
    r = 6371.0
    phi1 = np.deg2rad(lat1)
    phi2 = np.deg2rad(lat2)
    dphi = np.deg2rad(lat2 - lat1)
    dlmb = np.deg2rad(lon2 - lon1)

    a = np.sin(dphi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * (np.sin(dlmb / 2.0) ** 2)
    return float(2.0 * r * np.arcsin(np.sqrt(a)))


def rule_based_match_id(
    cid_norm: str,
    cname_norm: str,
    caud_index: set[str],
) -> Optional[str]:
    """
    Apply ordered token rules to map a plant to a caudal series.
    Returns the matched series id if found and available, else None.
    """
    tokens = token_set(cid_norm) | token_set(cname_norm)

    for required_tokens, target_series in RULES_TOKEN_MATCH:
        if target_series not in caud_index:
            continue
        if all(t in tokens for t in required_tokens):
            return target_series

    # A softer version: any-token match (fallback), still ordered
    for required_tokens, target_series in RULES_TOKEN_MATCH:
        if target_series not in caud_index:
            continue
        if any(t in tokens for t in required_tokens):
            return target_series

    return None


def build_anchor_points(
    hydro_df: pd.DataFrame,
    caud_index: List[str],
    lat_col: str,
    lon_col: str,
) -> Dict[str, Tuple[float, float, str]]:
    """
    Build anchor coordinates for each caudal series using matching plant coordinates.
    Strategy:
    - Look for a hydro plant whose central_id_norm or central_norm equals the caudal series id.
    - If found and has valid coordinates, store as anchor.
    """
    anchors: Dict[str, Tuple[float, float, str]] = {}

    by_cid = hydro_df.set_index("central_id_norm", drop=False)
    by_name = hydro_df.set_index("central_norm", drop=False)

    for series_id in caud_index:
        row = None
        if series_id in by_cid.index:
            row = by_cid.loc[series_id]
        elif series_id in by_name.index:
            row = by_name.loc[series_id]

        if row is None:
            continue

        # If multiple rows match, take the first
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]

        lat = row.get(lat_col, np.nan)
        lon = row.get(lon_col, np.nan)
        if pd.notna(lat) and pd.notna(lon):
            anchors[series_id] = (float(lat), float(lon), "from_plants_xlsx")

    return anchors


def nearest_anchor_match(
    lat: float,
    lon: float,
    anchors: Dict[str, Tuple[float, float, str]],
) -> Tuple[Optional[str], Optional[float], Optional[str], Optional[float]]:
    """
    Match by nearest anchor. Returns:
    - best_series_id
    - best_distance_km
    - second_best_series_id
    - second_best_distance_km
    """
    if not anchors:
        return None, None, None, None

    distances = []
    for series_id, (alat, alon, _) in anchors.items():
        d = haversine_km(lat, lon, alat, alon)
        distances.append((series_id, d))

    distances.sort(key=lambda x: x[1])
    best_id, best_d = distances[0]
    if len(distances) > 1:
        second_id, second_d = distances[1]
    else:
        second_id, second_d = None, None

    return best_id, best_d, second_id, second_d


def confidence_from_nearest(
    best_km: Optional[float],
    second_km: Optional[float],
    cfg: ModelConfig,
) -> str:
    """
    Heuristic confidence for nearest-anchor matching.
    """
    if best_km is None:
        return "low"

    gap = None if second_km is None else (second_km - best_km)

    if best_km <= cfg.nn_high_km and (gap is None or gap >= cfg.nn_gap_high_km):
        return "high"
    if best_km <= cfg.nn_med_km and (gap is None or gap >= cfg.nn_gap_med_km):
        return "medium"
    return "low"


# =========================
# Main pipeline
# =========================
def build_pmax_hourly(
    plants_xlsx: Path,
    caudals_csv: Path,
    year: int,
    cfg: ModelConfig,
    aliases_csv: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    For each hydro plant (run-of-river or reservoir), build hourly time series:
    - p_max_pu (0..1)
    - p_max_mw (MW), using Potencia_Efectiva_MW when available

    Matching order (best effort):
      1) Manual alias file
      2) Rule-based token match (system heuristics)
      3) Exact match by central_id
      4) Exact match by plant name
      5) Fuzzy match by central_id
      6) Fuzzy match by plant name
      7) Nearest anchor match (using plant coordinates)

    Returns:
        pmax_pu: DataFrame [hours x Central_id]
        pmax_mw: DataFrame [hours x Central_id]
        match_report: DataFrame mapping each plant to a caudal series and matching method
    """
    plants = pd.read_excel(plants_xlsx)
    plants.columns = [str(c).strip() for c in plants.columns]

    col_cid = find_col(plants, "central_id")
    col_central = find_col(plants, "central")
    col_tipo = find_col(plants, "tipo_de_central")
    col_subtipo = find_col(plants, "subtipo_de_central")
    col_p_eff = find_col(plants, "potencia_efectiva_mw")
    col_tech = find_col(plants, "technology")

    # Latitude/Longitude column names vary across files
    col_lat = find_first_col(plants, ["latitud", "latitude", "lat"])
    col_lon = find_first_col(plants, ["longitud", "longitude", "lon", "lng"])
    if col_lat is None or col_lon is None:
        raise KeyError(
            "Could not find latitude/longitude columns in POWERPLANTS.xlsx. "
            "Expected something like Latitud/Longitud or Latitude/Longitude."
        )

    hydro = plants[plants.apply(lambda r: is_hydro_row(r, col_tipo, col_tech), axis=1)].copy()

    hydro["central_id_norm"] = hydro[col_cid].map(normalize_id)
    hydro["central_norm"] = hydro[col_central].map(normalize_id)
    hydro["subtipo_norm"] = hydro[col_subtipo].map(normalize_id)

    # Ensure numeric coordinates
    hydro[col_lat] = pd.to_numeric(hydro[col_lat], errors="coerce")
    hydro[col_lon] = pd.to_numeric(hydro[col_lon], errors="coerce")

    caud = parse_caudals(Path(caudals_csv))
    candidates = caud.index.tolist()
    caud_index_set = set(candidates)

    aliases = load_aliases(Path(aliases_csv))

    # Build anchor points for nearest-neighbor matching
    anchors = build_anchor_points(hydro, candidates, lat_col=col_lat, lon_col=col_lon)
    if anchors:
        logger.info(f"Built {len(anchors)} anchor points from POWERPLANTS.xlsx for nearest-neighbor matching.")
    else:
        logger.warning("No anchor points could be built; nearest-neighbor matching will be disabled.")

    hourly_idx = pd.date_range(
        f"{year}-01-01 00:00:00",
        f"{year}-12-31 23:00:00",
        freq="h",
    )
    pmax_pu = pd.DataFrame(index=hourly_idx)
    pmax_mw = pd.DataFrame(index=hourly_idx)

    match_rows = []

    for _, r in hydro.iterrows():
        cid = str(r[col_cid])
        cid_norm = r["central_id_norm"]
        c_name_norm = r["central_norm"]
        subtipo = r["subtipo_norm"]

        lat = r.get(col_lat, np.nan)
        lon = r.get(col_lon, np.nan)

        caudal_id = None
        method = None
        score = 0.0

        nn_best_km = None
        nn_second_km = None
        nn_second_id = None
        confidence = "low"

        # 1) Manual alias
        if cid_norm in aliases and aliases[cid_norm] in caud_index_set:
            caudal_id = aliases[cid_norm]
            method = "alias_manual"
            score = 1.0
            confidence = "high"

        # 2) Rule-based token match
        if caudal_id is None:
            rb = rule_based_match_id(cid_norm, c_name_norm, caud_index_set)
            if rb is not None:
                caudal_id = rb
                method = "rule_tokens"
                score = 0.95
                confidence = "high"

        # 3) Exact match by central_id
        if caudal_id is None and cid_norm in caud_index_set:
            caudal_id = cid_norm
            method = "exact_central_id"
            score = 1.0
            confidence = "high"

        # 4) Exact match by plant name
        if caudal_id is None and c_name_norm in caud_index_set:
            caudal_id = c_name_norm
            method = "exact_plant_name"
            score = 1.0
            confidence = "high"

        # 5) Fuzzy match by central_id
        if caudal_id is None:
            m, s = best_fuzzy_match(cid_norm, candidates, cutoff=cfg.fuzzy_cutoff)
            if m is not None:
                caudal_id, method, score = m, "fuzzy_central_id", s
                confidence = "medium" if s >= 0.85 else "low"

        # 6) Fuzzy match by plant name
        if caudal_id is None:
            m, s = best_fuzzy_match(c_name_norm, candidates, cutoff=cfg.fuzzy_cutoff)
            if m is not None:
                caudal_id, method, score = m, "fuzzy_plant_name", s
                confidence = "medium" if s >= 0.85 else "low"

        # 7) Nearest anchor match (only if still unmatched and coordinates exist)
        if caudal_id is None and pd.notna(lat) and pd.notna(lon) and anchors:
            best_id, best_km, second_id, second_km = nearest_anchor_match(float(lat), float(lon), anchors)
            if best_id is not None:
                caudal_id = best_id
                method = "nearest_anchor"
                score = 0.80  # heuristic score placeholder
                nn_best_km = best_km
                nn_second_id = second_id
                nn_second_km = second_km
                confidence = confidence_from_nearest(best_km, second_km, cfg)

        match_rows.append(
            {
                "central_id": cid,
                "central_id_norm": cid_norm,
                "plant_name_norm": c_name_norm,
                "matched_caudal_id": caudal_id,
                "method": method if method else "unmatched",
                "score": score,
                "confidence": confidence,
                "nn_best_km": nn_best_km,
                "nn_second_id": nn_second_id,
                "nn_second_km": nn_second_km,
                "subtipo_de_central": str(r[col_subtipo]),
                "potencia_efectiva_mw": r[col_p_eff],
                "lat": float(lat) if pd.notna(lat) else np.nan,
                "lon": float(lon) if pd.notna(lon) else np.nan,
            }
        )

        if caudal_id is None:
            continue

        Qm = caud.loc[caudal_id, [str(m) for m in range(1, 13)]].astype(float)
        Qm.index = pd.Index(range(1, 13), name="month")

        P_eff = float(r[col_p_eff]) if pd.notna(r[col_p_eff]) else np.nan
        if not np.isfinite(P_eff) or P_eff <= 0:
            P_eff = np.nan

        if subtipo == "pasada":
            p_month = pmax_pu_pasada(Qm, cfg)
        elif subtipo == "embalse":
            p_month = pmax_pu_embalse(Qm, cfg)
        else:
            p_month = pmax_pu_pasada(Qm, cfg)

        p_hour = monthly_to_hourly(p_month, year, cfg)

        pmax_pu[cid] = p_hour
        pmax_mw[cid] = p_hour * P_eff if np.isfinite(P_eff) else np.nan

    match_df = pd.DataFrame(match_rows)
    return pmax_pu, pmax_mw, match_df


def main() -> None:
    pmax_pu, pmax_mw, match_df = build_pmax_hourly(
        plants_xlsx=Path(POWERPLANTS_XLSX),
        caudals_csv=Path(CAUDALS_CSV),
        year=YEAR,
        cfg=CFG,
        aliases_csv=ALIASES_CSV,
    )

    out_pu = OUTPUT_DIR / f"pmax_pu_hydro_{YEAR}_hourly.csv"
    out_mw = OUTPUT_DIR / f"pmax_mw_hydro_{YEAR}_hourly.csv"
    out_match = OUTPUT_DIR / "match_report.csv"
    out_unmatched = OUTPUT_DIR / "unmatched_plants.csv"
    out_lowconf = OUTPUT_DIR / "low_confidence_matches.csv"

    pmax_pu.to_csv(out_pu, index=True)
    pmax_mw.to_csv(out_mw, index=True)
    match_df.to_csv(out_match, index=False)
    match_df[match_df["method"] == "unmatched"].to_csv(out_unmatched, index=False)
    match_df[match_df["confidence"] == "low"].to_csv(out_lowconf, index=False)

    logger.info(f"Saved pmax_pu: {out_pu}")
    logger.info(f"Saved pmax_mw: {out_mw}")
    logger.info(f"Saved match report: {out_match}")
    logger.info(f"Saved unmatched list: {out_unmatched}")
    logger.info(f"Saved low-confidence list: {out_lowconf}")

    # If no aliases file exists, create a template the user can fill
    if not ALIASES_CSV.exists():
        template = match_df[["central_id", "matched_caudal_id"]].copy()
        template["caudal_id"] = template["matched_caudal_id"].fillna("")
        template = template[["central_id", "caudal_id"]]
        template.to_csv(ALIASES_CSV, index=False)
        logger.info(f"Created aliases template: {ALIASES_CSV}")

    # Also export the anchors used (for auditing)
    try:
        plants = pd.read_excel(Path(POWERPLANTS_XLSX))
        plants.columns = [str(c).strip() for c in plants.columns]
        col_cid = find_col(plants, "central_id")
        col_central = find_col(plants, "central")
        col_lat = find_first_col(plants, ["latitud", "latitude", "lat"])
        col_lon = find_first_col(plants, ["longitud", "longitude", "lon", "lng"])
        plants["central_id_norm"] = plants[col_cid].map(normalize_id)
        plants["central_norm"] = plants[col_central].map(normalize_id)

        caud = parse_caudals(Path(CAUDALS_CSV))
        hydro = plants.copy()
        hydro[col_lat] = pd.to_numeric(hydro[col_lat], errors="coerce")
        hydro[col_lon] = pd.to_numeric(hydro[col_lon], errors="coerce")

        anchors = build_anchor_points(hydro, caud.index.tolist(), lat_col=col_lat, lon_col=col_lon)
        anchor_df = pd.DataFrame(
            [{"caudal_id": k, "lat": v[0], "lon": v[1], "source": v[2]} for k, v in anchors.items()]
        )
        anchor_df.to_csv(OUTPUT_DIR / "anchors_used.csv", index=False)
    except Exception as e:
        logger.warning(f"Could not export anchors_used.csv due to: {e}")


if __name__ == "__main__":
    main()


2026-01-08 11:55:21: Built 4 anchor points from POWERPLANTS.xlsx for nearest-neighbor matching.
2026-01-08 11:55:23: Saved pmax_pu: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\hidro_max_profiles\pmax_pu_hydro_2022_hourly.csv
2026-01-08 11:55:23: Saved pmax_mw: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\hidro_max_profiles\pmax_mw_hydro_2022_hourly.csv
2026-01-08 11:55:23: Saved match report: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\hidro_max_profiles\match_report.csv
2026-01-08 11:55:23: Saved unmatched list: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\hidro_max_profiles\unmatched_plants.csv
2026-01-08 11:55:23: Saved low-confidence list: c:\Repositories\Repos\pypsa-earth-project\EcuadorElectricGrid\data\processed\generation\hidro_max_profiles\low_confidence_matches.csv
